In [1]:
import os
import pandas as pd
import duckdb
import polars as pl
import re

In [2]:
os.getcwd()
os.listdir()
# for changing the directory
os.chdir('C:\\Users\\aise\\Documents\\julialkmlproject')

In [3]:
con = duckdb.connect()
path = "C:\\Users\\aise\\Documents\\julialkmlproject\\data.parquet"
con.execute(f"SELECT COUNT(*) FROM read_parquet('{path}')").fetchone()

(6233433,)

In [4]:
# getting the first 4 messages from lkml
query = """
SELECT *
FROM read_parquet('{}')
LIMIT 4

""".format(path)

lkml_messages = con.execute(query).fetchdf()
lkml_messages.head()

,from,to,cc,subject,date,client-date,message-id,in-reply-to,references,x-mailing-list,trailers,code,raw_body,__file_name
0,4ccf57469dd98f9074b30d463ada5fab6844ca8b,[199a11af4fdad9bcc673e7bba082bbfc36ada1e7],[f788566a4ef114d663f963f9fd5bf61269f6750b],[-mm patch] drivers/firewire/: cleanups,NaT,[2007-01-22T19:17:37+01:00],20070122181736.GU9093@stusta.de,20070111222627.66bb75ab.akpm@osdl.org,[20070111222627.66bb75ab.akpm@osdl.org],linux-kernel@vger.kernel.org,"[{'attribution': 'Signed-off-by', 'identificat...",[---\n\n drivers/firewire/fw-ohci.c | ...,"On Thu, Jan 11, 2007 at 10:26:27PM -0800, Andr...",0000000001-e0-b67bf7f62c8125d67461cc6e7d1736dd...
1,1767232d16d92cbbbdb29dacc57f46bbd3ac0f3d,[cdb47d8cfb6103d429b19e3fcc9008a1ad9e7540],[551a2101f3d6a25dc9b2bd0f336063241d01d6af],Re: [Announce] GIT v1.5.0-rc2,NaT,[2007-01-22T10:08:59-08:00],87lkjvhr2c.wl%cworth@cworth.org,7v3b6439uh.fsf@assigned-by-dhcp.cox.net,"[7v64b04v2e.fsf@assigned-by-dhcp.cox.net,, 7v3...",linux-kernel@vger.kernel.org,[],<NA>,"On Sun, 21 Jan 2007 03:20:06 -0800, Junio C Ha...",0000000002-e0-8e73d15569f5a0577a8fa73f87817b47...
2,d0fa2019acdaea14dc2fb43870bc59fbf0abb89d,[4da0ab0aa2f17d8bec339110b94a9192c2077296],[22db0aab690eb7973d40068683ee084df81ec738],Re: [PATCH] nfs: fix congestion control -v3,NaT,[2007-01-22T09:59:27-08:00],Pine.LNX.4.64.0701220956590.24578@schroedinger...,1169276500.6197.159.camel@twins,[20070116054743.15358.77287.sendpatchset@schro...,linux-kernel@vger.kernel.org,"[{'attribution': 'Acked-by', 'identification':...",<NA>,"On Sat, 20 Jan 2007, Peter Zijlstra wrote:\n\n...",0000000003-e0-19e22837568948d161505606336628cc...
3,446423177a483d46b64a39ac28195bf38ea9c6b4,[aba7ddeb68a57fe22fb189d556856df6d34a06e2],[3b9ad5d7789d71701e91c91213255e33599cb484],Re: [PATCH] Remove final reference to superflu...,NaT,[2007-01-22T18:57:50+01:00],1169488670.15515.17.camel@earth,Pine.LNX.4.64.0701201326330.24479@CPE00045a9c3...,[Pine.LNX.4.64.0701201326330.24479@CPE00045a9c...,linux-kernel@vger.kernel.org,"[{'attribution': 'Acked-by', 'identification':...",<NA>,"On Sat, 2007-01-20 at 13:28 -0500, Robert P. J...",0000000004-e0-02aaab890ccf685cda0a9e618c443847...


In [5]:
df_first = con.execute(f"""
    SELECT date, subject, "from", raw_body
    FROM read_parquet('{path}')
    ORDER BY date ASC
    LIMIT 4
""").fetchdf()

df_first

# print(df_first["raw_body"].iloc[0])


,date,subject,from,raw_body
0,2018-06-05 14:53:56,Re: linux-next: build warning after merge of t...,c45192582cd16134f910dfbc5688691fcee4f857,"On Mon, 4 Jun 2018 at 20:18, Stephen Rothwell ..."
1,2018-06-06 19:38:09,Re: [PATCH v4 2/3] arm: shmobile: Add the R9A0...,7ae68576e9181781a9685c35de3684bdb2f07069,"On 06/06/2018 12:30 PM, Frank Rowand wrote:\n>..."
2,2018-06-06 19:38:17,[PATCH 1/3] MIPS: jz4780: Allow access to jz47...,21734de582beb15f09fb0a778e2119e07fc7dc9b,Make it possible to select SND_JZ4740_SOC_I2S ...
3,2018-06-06 19:38:19,[PATCH 2/3] MIPS: Ci20: Enable SND_JZ4740_SOC ...,21734de582beb15f09fb0a778e2119e07fc7dc9b,Update the Ci20's defconfig to enable the JZ47...


In [6]:
con.execute(f"""
    SELECT min(date), max(date)
    FROM read_parquet('{path}')
""").fetchdf()


,min(date),max(date)
0,2018-06-05 14:53:56,2026-05-03 20:17:10


In [7]:
# testing authors parquet given by the dataset team
df= pl.read_parquet("id_map.parquet")
print(df)

shape: (6_233_433, 2)
┌─────────────────────────────────┬─────────────────────────────────┐
│ __original_from                 ┆ from                            │
│ ---                             ┆ ---                             │
│ str                             ┆ str                             │
╞═════════════════════════════════╪═════════════════════════════════╡
│ Adrian Bunk <bunk@stusta.de>    ┆ 4ccf57469dd98f9074b30d463ada5f… │
│ Carl Worth <cworth@cworth.org>  ┆ 1767232d16d92cbbbdb29dacc57f46… │
│ Christoph Lameter <clameter@sg… ┆ d0fa2019acdaea14dc2fb43870bc59… │
│ Ingo Molnar <mingo@redhat.com>  ┆ 446423177a483d46b64a39ac28195b… │
│ Björn Steinbrink <B.Steinbrink… ┆ dfadf4db90e2a6b33224b07106c0d7… │
│ …                               ┆ …                               │
│ Nitin Joshi <nitjoshi@gmail.co… ┆ 3710ba1b229eed886c5192c6a8c0f5… │
│ Baokun Li <libaokun1@huawei.co… ┆ bc7b93c723b495d7955cded8ea0304… │
│ Al Viro <viro@zeniv.linux.org.… ┆ 72ccface49dd48d15c932f53d49481… 

In [8]:
AUTHOR_MAP_PATH = (
    "id_map.parquet"
)

OUTPUT_PATH = (
    "email_alias_table_lkml.csv"
)

AUTHORS = {
    "Greg Kroah-Hartman": [
        "kroah",
        "greg kh",
        "greg kroah-hartman",
    ],

     "Linus Torvalds": [
        "linus torvalds",
        "linus t0rvalds",
        "linus torv",
    ],
}

BAD_EMAIL_PATTERNS = [
    "tip-bot",
    "tipbot",
    "helpdesk",
    "noreply",
    "no-reply",
    "daemon",
    "griffin"
]


def extract_email(s):
    if not s:
        return None

    m = re.search(r"<([^>]+)>", s)

    if m:
        return m.group(1).strip().lower()

    s2 = s.strip().lower()

    return s2 if "@" in s2 and " " not in s2 else None


def is_valid_email(email):
    if not email:
        return False

    email = email.lower()

    for pattern in BAD_EMAIL_PATTERNS:
        if pattern in email:
            return False

    return True


def build_author_table(df, sender_name, patterns):
    condition = None

    for p in patterns:
        expr = pl.col("from_raw_lc").str.contains(p)

        if condition is None:
            condition = expr
        else:
            condition = condition | expr

    return (
        df
        .filter(condition)
        .with_columns(
            pl.lit(sender_name).alias("Sender")
        )
    )


# load aliases
authors = (
    pl.read_parquet(AUTHOR_MAP_PATH)
    .with_columns([
        pl.col("__original_from")
        .fill_null("")
        .alias("from_raw"),

        pl.col("__original_from")
        .fill_null("")
        .str.to_lowercase()
        .alias("from_raw_lc"),

        pl.col("__original_from")
        .map_elements(
            extract_email,
            return_dtype=pl.Utf8
        )
        .alias("email")
    ])
)

# build author subsets
author_tables = []

for sender, patterns in AUTHORS.items():
    tbl = build_author_table(authors, sender, patterns)
    author_tables.append(tbl)

authors_filtered = pl.concat(author_tables)

# clean emails
authors_filtered = (
    authors_filtered
    .filter(
        pl.col("email").map_elements(
            is_valid_email,
            return_dtype=pl.Boolean
        )
    )
)

# unique ids
all_ids = (
    authors_filtered
    .select(pl.col("from").unique())
    .to_series()
    .to_list()
)

print("Unique author IDs:", len(all_ids))

# id -> email mapping
id_to_email = (
    authors_filtered
    .select([
        "Sender",
        "from",
        "email",
        "__original_from"
    ])
    .unique()
)

# count messages
con = duckdb.connect()

counts = con.execute(f"""
    SELECT
        "from" AS author_id,
        COUNT(*) AS msg_count
    FROM read_parquet('{path}')
    WHERE "from" IS NOT NULL
    GROUP BY "from"
""").fetchdf()

counts_pl = pl.from_pandas(counts)

# keep only selected authors
counts_pl = counts_pl.filter(
    pl.col("author_id").is_in(all_ids)
)

# aggregate counts
out = (
    id_to_email
    .rename({"from": "author_id"})
    .join(counts_pl, on="author_id", how="inner")
    .group_by(["Sender", "email"])
    .agg(
        pl.col("msg_count")
        .sum()
        .alias("Count")
    )
    .sort(
        ["Sender", "Count"],
        descending=[False, True]
    )
)

# save
out.write_csv(OUTPUT_PATH)

print(out)
print(f"Saved to: {OUTPUT_PATH}")

Unique author IDs: 54
shape: (20, 3)
┌────────────────────┬─────────────────────────────┬────────┐
│ Sender             ┆ email                       ┆ Count  │
│ ---                ┆ ---                         ┆ ---    │
│ str                ┆ str                         ┆ i64    │
╞════════════════════╪═════════════════════════════╪════════╡
│ Greg Kroah-Hartman ┆ gregkh@linuxfoundation.org  ┆ 297740 │
│ Greg Kroah-Hartman ┆ gregkh@suse.de              ┆ 24238  │
│ Greg Kroah-Hartman ┆ greg@kroah.com              ┆ 17851  │
│ Greg Kroah-Hartman ┆ greg@wirex.com              ┆ 50     │
│ Greg Kroah-Hartman ┆ gregkh@linux-foundation.org ┆ 46     │
│ …                  ┆ …                           ┆ …      │
│ Linus Torvalds     ┆ linus@torvalds.org          ┆ 3      │
│ Linus Torvalds     ┆ linus@linuxfoundation.org   ┆ 1      │
│ Linus Torvalds     ┆ tordalds@transmeta.com      ┆ 1      │
│ Linus Torvalds     ┆ t0rvalds@transmeta.com      ┆ 1      │
│ Linus Torvalds     ┆ linus@linu

In [9]:
def extract_email(s):
    if not s:
        return None
    m = re.search(r"<([^>]+)>", s)
    if m:
        return m.group(1).strip().lower()
    s2 = s.strip().lower()
    return s2 if "@" in s2 and " " not in s2 else None

In [10]:
email_alias_df = pl.read_csv("C:\\Users\\aise\\Documents\\julialkmlproject\\email_alias_table_lkml.csv")

linus_emails = email_alias_df.filter(pl.col("Sender") == "Linus Torvalds")["email"].to_list()
greg_emails = email_alias_df.filter(pl.col("Sender") == "Greg Kroah-Hartman")["email"].to_list()

print(f"Emails do Linus: {linus_emails}")
print(f"Emails do Greg: {greg_emails}")

Emails do Linus: ['torvalds@linux-foundation.org', 'torvalds@osdl.org', 'torvalds@transmeta.com', 'torvalds@linuxfoundation.org', 'linus@torvalds.org', 'linus@linuxfoundation.org', 'tordalds@transmeta.com', 't0rvalds@transmeta.com', 'linus@linux-foundation.org']
Emails do Greg: ['gregkh@linuxfoundation.org', 'gregkh@suse.de', 'greg@kroah.com', 'greg@wirex.com', 'gregkh@linux-foundation.org', 'gregkh@kernel.org', 'gregkh@us.ibm.com', 'gregkh@google.com', 'og@kroah.com', 'gregkh@linux.com', 'gregkh@linuxfoundtion.org']


In [11]:
mapping_path = "C:\\Users\\aise\\Documents\\julialkmlproject\\id_map.parquet"

mapping_df = pl.read_parquet(mapping_path).with_columns(
    pl.col("__original_from").map_elements(extract_email, return_dtype=pl.Utf8).alias("email")
)

linus_ids = mapping_df.filter(pl.col("email").is_in(linus_emails))["from"].unique().to_list()
greg_ids = mapping_df.filter(pl.col("email").is_in(greg_emails))["from"].unique().to_list()

print(f"IDs do Linus: {len(linus_ids)}")
print(f"IDs do Greg: {len(greg_ids)}")

IDs do Linus: 12
IDs do Greg: 56


In [12]:
con = duckdb.connect()
path = "C:\\Users\\aise\\Documents\\julialkmlproject\\data.parquet"

linus_df = con.execute(f"""
    SELECT *
    FROM read_parquet('{path}')
    WHERE "from" IN {tuple(linus_ids) if len(linus_ids) > 1 else f"('{linus_ids[0]}')"}
""").fetchdf()

greg_all = con.execute(f"""
    SELECT *
    FROM read_parquet('{path}')
    WHERE "from" IN {tuple(greg_ids) if len(greg_ids) > 1 else f"('{greg_ids[0]}')"}
""").fetchdf()

print(f"Mensagens do Linus: {len(linus_df)}")
print(f"Mensagens do Greg (total): {len(greg_all)}")

Mensagens do Linus: 32890
Mensagens do Greg (total): 340921


In [13]:
if len(greg_all) > 32885:
    greg_sample = greg_all.sample(n=32885, random_state=42)
else:
    greg_sample = greg_all

print(f"Mensagens do Greg (amostradas): {len(greg_sample)}")

Mensagens do Greg (amostradas): 32885


In [14]:
all_authors_df = pd.concat([linus_df, greg_all], ignore_index=True)
print(f"Total de mensagens (todos os autores): {len(all_authors_df)}")

all_authors_df.to_csv("C:\\Users\\aise\\Documents\\julialkmlproject\\all_messages.csv", index=False)
print("Salvo em: all_messages.csv")

Total de mensagens (todos os autores): 373811
Salvo em: all_messages.csv


In [15]:
final_df = pd.concat([linus_df, greg_sample], ignore_index=True)
print(f"Total de mensagens (amostrado): {len(final_df)}")

final_df.to_csv("C:\\Users\\aise\\Documents\\julialkmlproject\\messages_output.csv", index=False)
print("Salvo em: messages_output.csv")

Total de mensagens (amostrado): 65775
Salvo em: messages_output.csv


In [16]:
linus_ids_set = set(linus_ids)
greg_ids_set = set(greg_ids)

linus_counts = final_df[final_df["from"].isin(linus_ids_set)].groupby("from").size().reset_index(name="msg_count")
greg_counts = final_df[final_df["from"].isin(greg_ids_set)].groupby("from").size().reset_index(name="msg_count")

linus_mapping = mapping_df.filter(pl.col("email").is_in(linus_emails)).select(["email", "from"]).unique()
greg_mapping = mapping_df.filter(pl.col("email").is_in(greg_emails)).select(["email", "from"]).unique()

linus_summary = linus_mapping.to_pandas().merge(linus_counts, left_on="from", right_on="from")
linus_summary = linus_summary.groupby(["email"], as_index=False)["msg_count"].sum()
linus_summary["sender"] = "Linus Torvalds"

greg_summary = greg_mapping.to_pandas().merge(greg_counts, left_on="from", right_on="from")
greg_summary = greg_summary.groupby(["email"], as_index=False)["msg_count"].sum()
greg_summary["sender"] = "Greg Kroah-Hartman"

combined_counts = pd.concat([linus_summary, greg_summary], ignore_index=True)
combined_counts = combined_counts.sort_values(["sender", "msg_count"], ascending=[True, False])
combined_counts = combined_counts[["email", "sender", "msg_count"]]
combined_counts.to_csv("C:\\Users\\aise\\Documents\\julialkmlproject\\author_message_counts.csv", index=False)

print("Arquivos gerados:")
print(f"  - messages_output.csv: {len(final_df)} mensagens")
print(f"  - all_messages.csv: {len(all_authors_df)} mensagens")
print(f"  - author_message_counts.csv: {len(combined_counts)} emails")
print(f"\nResumo por sender:")
print(combined_counts.groupby("sender")["msg_count"].sum())
print(linus_ids_set)
print(greg_ids_set)

Arquivos gerados:
  - messages_output.csv: 65775 mensagens
  - all_messages.csv: 373811 mensagens
  - author_message_counts.csv: 15 emails

Resumo por sender:
sender
Greg Kroah-Hartman    32885
Linus Torvalds        32890
Name: msg_count, dtype: int64
{'a97dfd08c1662803a39537e7e42c93bf2db27128', 'd04d508bd126c72cc5eb1b81ab25ef19678060be', '3507a1d944456b02389f26658705675c228d3c1a', '3bd208ad71e484c15efd05cf4e5f5c46d77168c9', '8dc38ed98e552fdf8ed1c2cda11d2bba43c467f0', 'd4db8227a2f6acd9396e2f96a574416cbbf86304', '38d186e8d1752771441f67080ca38409d807c50a', 'a86865bfe5b8f588be22e0f968b86ca05809ca1d', 'e7ee4d5fa7b3eb2fbb1005393cee4d3f8f0aa74f', '49e50779221eb10c41e9309c6613755da56299c8', '07ab1331b0fbe2deb4a8e5349795bfeed7480588', 'f5c9a22d72bffbfb7c3c6690db9a8982ab1c57ac'}
{'33b4687a1bce892dd50a292f1aef7123d34e1e80', '430a69d897a421da23b624bcb40ad627abc7e441', 'e70a43732b757912deeedad36816c065f0e39c9d', '7837fbaa206929b8f395922b48b1fc9e03378645', 'aa271e70d86fad5d6665d697f8baf9716f1ae5f4'